In [ ]:
# ------------------------------------------------------------
# Data Loading
# ------------------------------------------------------------
# Purpose: Load raw Ubuntu Dialogue Corpus files into DataFrames.


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the in…

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
# ------------------------------------------------------------
# Setup
# ------------------------------------------------------------
# Purpose: Import libraries and set notebook configuration.

import sys
import time
import random
import statistics
from IPython.display import Image, display

import re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
# Adjust pandas display option to show full text in columns
pd.set_option('display.max_colwidth', None)

import warnings
warnings.filterwarnings("ignore")


# Print library versions with consistent spacing
print(f"{'NumPy version:':<25} {np.__version__}")
print(f"{'Pandas version:':<25} {pd.__version__}")
print(f"{'Matplotlib version:':<25} {matplotlib.__version__}")

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

input_directory = '/kaggle/input'

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

def inspect_directory(directory_path):
# Inspects a directory and creates a mapping of filenames (without extensions) to their full file paths.
    file_mapping = {}

    for current_directory, _, file_names in os.walk(directory_path):
        for file_name in file_names:
            file_path = os.path.join(current_directory, file_name)
            if file_name.split('.')[0] != '':
                base_file_name = file_name.split('.')[0]
                file_mapping[base_file_name] = file_path

    return file_mapping

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

file_mapping = inspect_directory(input_directory)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

file_mapping

In [ ]:
# ------------------------------------------------------------
# Data Loading
# ------------------------------------------------------------
# Purpose: Load raw Ubuntu Dialogue Corpus files into DataFrames.

toc = pd.read_csv(file_mapping['toc'])
toc

In [ ]:
# ------------------------------------------------------------
# Data Loading
# ------------------------------------------------------------
# Purpose: Load raw Ubuntu Dialogue Corpus files into DataFrames.

dialogueText = pd.read_csv(file_mapping['dialogueText'])
dialogueText.head()

In [ ]:
# ------------------------------------------------------------
# Data Loading
# ------------------------------------------------------------
# Purpose: Load raw Ubuntu Dialogue Corpus files into DataFrames.

dialogueText_196 = pd.read_csv(file_mapping['dialogueText_196'])
dialogueText_196.head()

In [ ]:
# ------------------------------------------------------------
# Data Loading
# ------------------------------------------------------------
# Purpose: Load raw Ubuntu Dialogue Corpus files into DataFrames.

dialogueText_301 = pd.read_csv(file_mapping['dialogueText_301'])
dialogueText_301.head()

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

print(f"{toc['words'].sum()/10**6:.2f} million")

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText_turns = len(dialogueText)
dialogueText_turns

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText_301_turns = len(dialogueText_301)
dialogueText_301_turns

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText_196_turns = len(dialogueText_196)
dialogueText_196_turns

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

total_corpus_turns = dialogueText_turns + dialogueText_301_turns + dialogueText_196_turns
print(f'Total_corpus_turns is : {total_corpus_turns/10**6}  million.')

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText.info()

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText['date'].iloc[0]
# Inspect date formate.

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

def convert_date_column(df, date_column='date'):
# Convert the specified date column in a DataFrame from object to datetime.
    df[date_column] = pd.to_datetime(df[date_column], format='%Y-%m-%dT%H:%M:%S.%fZ', errors='coerce')
    return df

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText = convert_date_column(dialogueText)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText.info()

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText_301 = convert_date_column(dialogueText_301)
dialogueText_196 = convert_date_column(dialogueText_196)

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

len(dialogueText['folder'].value_counts())

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

len(dialogueText_301['folder'].value_counts())

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

len(dialogueText_196['folder'].value_counts())

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

num_floders = {'dialogueText': 1, 'dialogueText_196': 173, 'dialogueText_301': 349}

files = list(num_floders.keys())
values = list(num_floders.values())

# Create the bar plot
plt.bar(files, values, color='g')
plt.show()

In [ ]:
# ------------------------------------------------------------
# Dialogue Processing
# ------------------------------------------------------------
# Purpose: Filter, validate, and aggregate messages into dialogues.

def analysis_per_dialogueID(df, dialogue_column='dialogueID', from_column='from', to_column='to', date_column='date'):
# Analyzes the dialogues in the DataFrame by grouping by a specified dialogue ID and calculating various metrics,
    grouped = df.groupby(dialogue_column)

    analysis = grouped.agg({
        from_column: lambda x: list(x.unique()),  # Unique 'from' values
        to_column: lambda x: list(x.unique()),    # Unique 'to' values
        date_column: lambda x: x.max() - x.min()  # Interval between max and min date
    }).reset_index()

    # Rename columns for clarity
    analysis.rename(columns={from_column: 'unique_from', to_column: 'unique_to', date_column: 'date_interval'}, inplace=True)

    # Create two new columns to store the counts of unique values in 'unique_from' and 'unique_to'
    analysis['unique_from_count'] = analysis['unique_from'].map(lambda x: len(x))
    analysis['unique_to_count'] = analysis['unique_to'].map(lambda x: len(x))


    return analysis

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

summary_per_dialogueID = analysis_per_dialogueID(dialogueText)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

summary_per_dialogueID.head()

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

print(summary_per_dialogueID['unique_from_count'].max())
print(summary_per_dialogueID['unique_from_count'].min())

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

print(summary_per_dialogueID['unique_to_count'].max())
print(summary_per_dialogueID['unique_to_count'].min())

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

summary_per_dialogueID_301 = analysis_per_dialogueID(dialogueText_301)
summary_per_dialogueID_196 = analysis_per_dialogueID(dialogueText_196)

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

summary_per_dialogueID_301[['unique_from_count']].describe()

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

summary_per_dialogueID_301[['unique_to_count']].describe()

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

summary_per_dialogueID_196[['unique_from_count']].describe()

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

summary_per_dialogueID_196[['unique_to_count']].describe()

In [ ]:
# ------------------------------------------------------------
# Dialogue Processing
# ------------------------------------------------------------
# Purpose: Filter, validate, and aggregate messages into dialogues.

def analysis_per_dialogueID_and_folder(df, dialogue_column='dialogueID', folder_column='folder', from_column='from', to_column='to', date_column='date'):
# Analyzes the dialogues in the DataFrame by grouping by both 'dialogueID' and 'folder', and calculating various metrics,
    # Group by both 'dialogueID' and 'folder' columns
    grouped = df.groupby([dialogue_column, folder_column])

    # Perform aggregation
    analysis = grouped.agg({
        from_column: lambda x: list(x.unique()),  # Unique 'from' values
        to_column: lambda x: list(x.unique()),    # Unique 'to' values
        date_column: lambda x: x.max() - x.min()  # Interval between max and min date
    }).reset_index()

    # Rename columns for clarity
    analysis.rename(columns={from_column: 'unique_from', to_column: 'unique_to', date_column: 'date_interval'}, inplace=True)

    # Create two new columns to store the counts of unique values in 'unique_from' and 'unique_to'
    analysis['unique_from_count'] = analysis['unique_from'].map(lambda x: len(x))
    analysis['unique_to_count'] = analysis['unique_to'].map(lambda x: len(x))

    return analysis

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

summary_per_dialogueID_and_folder_301 = analysis_per_dialogueID_and_folder(dialogueText_301)
summary_per_dialogueID_and_folder_196 = analysis_per_dialogueID_and_folder(dialogueText_196)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

print(summary_per_dialogueID_and_folder_301['unique_from_count'].max())
print(summary_per_dialogueID_and_folder_301['unique_from_count'].min())
print(summary_per_dialogueID_and_folder_301['unique_to_count'].max())
print(summary_per_dialogueID_and_folder_301['unique_to_count'].min())

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

print(summary_per_dialogueID_and_folder_196['unique_from_count'].max())
print(summary_per_dialogueID_and_folder_196['unique_from_count'].min())
print(summary_per_dialogueID_and_folder_196['unique_to_count'].max())
print(summary_per_dialogueID_and_folder_196['unique_to_count'].min())

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

def compare_dialogues_by_folder_and_id(df1, df2, folder_col='folder', id_col='dialogueID'):
# Compares two DataFrames to check if the same combination of `folder` and `dialogueID`
    # Extract the common combinations of folder and dialogueID
    common_combinations = pd.merge(
        df1[[folder_col, id_col]].drop_duplicates(),
        df2[[folder_col, id_col]].drop_duplicates(),
        on=[folder_col, id_col]
    )
    
    # Filter rows in both DataFrames based on the common combinations
    filtered_df1 = pd.merge(df1, common_combinations, on=[folder_col, id_col])
    filtered_df2 = pd.merge(df2, common_combinations, on=[folder_col, id_col])
    
    return filtered_df1, filtered_df2

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

def compare_dataframes(df1, df2):
# Compares two DataFrames to determine if they are identical.
    if df1.equals(df2):
        return "The two DataFrames are identical."
    else:
        return "The two DataFrames are not identical."

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText_common_dialogueText_301, dialogueText_301_common_dialogueText \
= compare_dialogues_by_folder_and_id(dialogueText, dialogueText_301)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

compare_dataframes(dialogueText_common_dialogueText_301, dialogueText_301_common_dialogueText)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText_common_dialogueText_196, dialogueText_196_common_dialogueText\
= compare_dialogues_by_folder_and_id(dialogueText, dialogueText_196)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

compare_dataframes(dialogueText_common_dialogueText_196, dialogueText_196_common_dialogueText)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText_301_common_dialogueText_196, dialogueText_196_common_dialogueText_301\
= compare_dialogues_by_folder_and_id(dialogueText_301, dialogueText_196)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

compare_dataframes(dialogueText_301_common_dialogueText_196, dialogueText_196_common_dialogueText_301)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

len(dialogueText_301_common_dialogueText_196)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

len(dialogueText_196_common_dialogueText_301)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

compare_dataframes(dialogueText_301_common_dialogueText_196[:-10], dialogueText_196_common_dialogueText_301)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText_301_common_dialogueText_196[-10:][['folder', 'dialogueID']].drop_duplicates()

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText_301[(dialogueText_301['folder'] == 13) 
& (dialogueText_301['dialogueID'] == '16586.tsv')]

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogueText_196[(dialogueText_196['folder'] == 13) 
& (dialogueText_196['dialogueID'] == '16586.tsv')].sort_values(by='date')

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

def concatenate_and_remove_duplicates(df_list):
# Concatenates a list of DataFrames along axis 0 and removes duplicate rows.
    # Concatenate DataFrames along axis 0
    combined_df = pd.concat(df_list, axis=0)
    
    # Remove duplicate rows
    combined_df = combined_df.drop_duplicates()
    
    return combined_df

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

full_corpus_data = concatenate_and_remove_duplicates([dialogueText, dialogueText_196, dialogueText_301])

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

full_corpus_data.head(10)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

full_corpus_data.info()

In [ ]:
# ------------------------------------------------------------
# Missing Value Handling
# ------------------------------------------------------------
# Purpose: Identify and handle missing / invalid records.

full_corpus_data.isnull().sum()

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

def get_random_dialogues_with_missing_value(df, column_name, num_samples=10, random_state=None):
# Returns random samples of entire dialogues where the specified column has missing values.
    # Step 1: Find rows where the specified column has missing values
    missing_value_rows = df[df[column_name].isna()]

    # Step 2: Find the unique combinations of `folder` and `dialogueID` that have missing values in the sp…
    missing_dialogues = missing_value_rows[['folder', 'dialogueID']].drop_duplicates()

    # Step 3: Sample random dialogue combinations
    if len(missing_dialogues) < num_samples:
        sampled_dialogues = missing_dialogues
    else:
        sampled_dialogues = missing_dialogues.sample(n=num_samples, random_state=None)

    # Step 4: Use merge to filter the DataFrame and get rows for the sampled dialogues
    sampled_dialogues_data = pd.merge(df, sampled_dialogues, on=['folder', 'dialogueID'], how='inner')

    return sampled_dialogues_data

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

get_random_dialogues_with_missing_value(full_corpus_data, 'text', 5)

In [ ]:
# ------------------------------------------------------------
# Missing Value Handling
# ------------------------------------------------------------
# Purpose: Identify and handle missing / invalid records.

full_corpus_data = full_corpus_data.dropna(subset='text').reset_index(drop=True)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

get_random_dialogues_with_missing_value(full_corpus_data, 'from', 2)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

summary_per_dialogue = analysis_per_dialogueID_and_folder(full_corpus_data)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

summary_per_dialogue.head()

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

summary_per_dialogue[['unique_from_count']].describe()

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

summary_per_dialogue[['unique_to_count']].describe()

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

def drop_invalid_dialogues(df, summary_per_dialogue, folder_col='folder', id_col='dialogueID'):
# Removes dialogues from the DataFrame where either the 'from' or 'to' fields have only 1 unique value,
    
    # Select valid dialogues where 'unique_from_count' and 'unique_to_count' are greater than 1
    filtered_summary_per_dialogue = summary_per_dialogue[
        (summary_per_dialogue['unique_from_count'] > 1) & 
        (summary_per_dialogue['unique_to_count'] > 1)
    ]
    
    # Filter the original DataFrame to include only valid dialogues
    filtered_df = df.merge(
        filtered_summary_per_dialogue[[folder_col, id_col]], 
        on=[folder_col, id_col], 
        how='inner'
    )

    # Calculate the number of dropped dialogues
    dropped_dialogues_count = len(df) - len(filtered_df)
    total_dialogues_count = len(df)

    # Calculate the percentage of dropped dialogues
    drop_percentage = (dropped_dialogues_count / total_dialogues_count) * 100

    # Return the filtered DataFrame, the filtered summary, and the drop percentage
    return filtered_df, filtered_summary_per_dialogue, drop_percentage

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

full_corpus_data, summary_per_dialogue, drop_percentage = drop_invalid_dialogues(full_corpus_data, summary_per_dialogue)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

print(f"Percentage of dropped dialogues: {drop_percentage:.2f}%")

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

full_corpus_data.head()

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

full_corpus_data.info()

In [ ]:
# ------------------------------------------------------------
# Text Cleaning / Standardization
# ------------------------------------------------------------
# Purpose: Normalize message text to improve training stability.

# Calculate the number of tokens (words) in each text turn
full_corpus_data['text_token_count'] = full_corpus_data['text'].apply(lambda x: len(str(x).split()))

# Preview the updated corpus dataset
full_corpus_data.head()

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

full_corpus_data[['text_token_count']].describe().T

In [ ]:
# ------------------------------------------------------------
# Text Cleaning / Standardization
# ------------------------------------------------------------
# Purpose: Normalize message text to improve training stability.

print(f"The number of turns with zero tokens are: {len(full_corpus_data[full_corpus_data['text_token_count']==0])}")

In [ ]:
# ------------------------------------------------------------
# Text Cleaning / Standardization
# ------------------------------------------------------------
# Purpose: Normalize message text to improve training stability.

# Calculate the number of rows with only white spaces in the 'text' column
white_space_rows = full_corpus_data['text'].apply(lambda x: str(x).strip() == '').sum()

print(f"Number of turns containing only white spaces: {white_space_rows}")

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

# Drop turns where the 'text' column contains only white spaces
full_corpus_data = full_corpus_data[~full_corpus_data['text'].apply(lambda x: str(x).strip() == '')]

# Verify the updated corpus dataset
full_corpus_data[['text_token_count']].describe().T

In [ ]:
# ------------------------------------------------------------
# Text Cleaning / Standardization
# ------------------------------------------------------------
# Purpose: Normalize message text to improve training stability.

# Get the minimum value of 'text_token_count'
min_text_token_count = full_corpus_data['text_token_count'].min()

# Get the maximum value of 'text_token_count'
max_text_token_count = full_corpus_data['text_token_count'].max()

print(f"Minimum text token count: {min_text_token_count}")
print(f"Maximum text token count: {max_text_token_count}")

In [ ]:
# ------------------------------------------------------------
# Text Cleaning / Standardization
# ------------------------------------------------------------
# Purpose: Normalize message text to improve training stability.

def clean_text_df(text):
# Cleans and preprocesses the input text for NLP tasks.
    if pd.isna(text):
        return text

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Replace underscores and hyphens with spaces
    text = re.sub(r'[_-]', ' ', text)

    # Remove non-alphanumeric characters (preserve whitespace)
    text = re.sub(r'[^a-zA-Z\s\d]', '', text)

    # Convert multiple whitespace characters to a single space
    text = re.sub(r'\s+', ' ', text)

    # Convert text to lowercase
    text = text.lower()

    # Reduce elongated words (e.g., "chooooose" to "choose")
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    return text

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

full_corpus_data['text'] = full_corpus_data['text'].map(clean_text_df)

In [ ]:
# ------------------------------------------------------------
# Text Cleaning / Standardization
# ------------------------------------------------------------
# Purpose: Normalize message text to improve training stability.

full_corpus_data['text_token_count'] = full_corpus_data['text'].apply(lambda x: len(str(x).split()))

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

full_corpus_data[['text_token_count']].describe().T

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

# Drop turns where the 'text' column contains zero tokens
full_corpus_data = full_corpus_data[~full_corpus_data['text'].apply(lambda x: str(x).strip() == '')]

# Verify the updated corpus dataset
full_corpus_data[['text_token_count']].describe().T

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

full_text = (full_corpus_data.text).to_list()
for i in full_text[:6]:
    print(i +'\n')

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

with open('full_text.txt', 'w') as f:
    for line in full_text:
        f.write(line + '\n')

In [ ]:
# ------------------------------------------------------------
# Dialogue Processing
# ------------------------------------------------------------
# Purpose: Filter, validate, and aggregate messages into dialogues.

# Group by 'folder' and 'dialogueID', sort within each group by 'date', and collect the 'text' column…
dialogues_corpus = (
    full_corpus_data
    .groupby(['folder', 'dialogueID'])
    .agg(dialogue_texts=('text', list))  # Aggregate the 'text' column into a list
    .reset_index()  # Reset index to turn grouped columns into regular columns
)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogues_corpus.sample(5)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

# This ensure that the grouping happen in the right way.
full_corpus_data[(full_corpus_data['folder']==6)&(full_corpus_data['dialogueID']=='31849.tsv')]

In [ ]:
# ------------------------------------------------------------
# Exploratory Data Analysis (EDA)
# ------------------------------------------------------------
# Purpose: Run lightweight sanity checks and inspect distributions.

dialogues_corpus['num_turns'] = dialogues_corpus['dialogue_texts'].map(len)  
dialogues_corpus[['num_turns']].describe().T

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

len(dialogues_corpus)

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

dialogues_corpus = dialogues_corpus[~(dialogues_corpus['num_turns'] == 1)]

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.

len(dialogues_corpus)

In [ ]:
# ------------------------------------------------------------
# Export / Save
# ------------------------------------------------------------
# Purpose: Persist cleaned outputs for downstream modeling (Part 2).

dialogues_corpus.to_csv('cleaned_dialogues_corpus.csv')

In [ ]:
# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------
# Purpose: Transform data step-by-step for model-ready inputs.
